# WSL + VS Code Environment Replication (EDO)

Current as of 2025-12-24.

This notebook exports your current Ubuntu WSL + VS Code setup and provides steps to recreate it on another Ubuntu WSL machine. Review each cell before running. Some steps require sudo and may be slow (TeX Live, Docker).


## How to use

-   Run Section A on the _source_ WSL machine to export settings, extensions, and package lists.
-   Copy the export folder to the _target_ WSL machine.
-   Run Sections B-G on the target machine.
-   Optional: Section H scaffolds a dev container for this repo.
-   Windows user settings are synced in Section E.


In [1]:
import os
from pathlib import Path

EXPORT_DIR = Path(os.environ.get("EXPORT_DIR", "wsl-env-export")).expanduser()
WORKSPACE_ROOT = Path(os.environ.get("WORKSPACE_ROOT", "/home/victor/projects/edo/main")).expanduser()
WIN_USER = os.environ.get("WIN_USER", "vic-l")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

os.environ["EXPORT_DIR"] = str(EXPORT_DIR)
os.environ["WORKSPACE_ROOT"] = str(WORKSPACE_ROOT)
os.environ["WIN_USER"] = WIN_USER

print("EXPORT_DIR:", EXPORT_DIR)
print("WORKSPACE_ROOT:", WORKSPACE_ROOT)
print("WIN_USER:", WIN_USER)


EXPORT_DIR: wsl-env-export
WORKSPACE_ROOT: /home/victor/projects/edo/main
WIN_USER: vic-l


## A. Export current WSL + VS Code (run on source machine)


In [2]:
%%bash
set -euo pipefail

: "${EXPORT_DIR:?}"
: "${WORKSPACE_ROOT:?}"
: "${WIN_USER:?}"

mkdir -p "$EXPORT_DIR/vscode" "$EXPORT_DIR/apt-sources"

{
  echo "## OS"
  cat /etc/os-release || true
  echo
  uname -a || true
  echo
  echo "## PATH"
  echo "$PATH"
  echo
  echo "## Versions"
  command -v node >/dev/null 2>&1 && node -v || true
  command -v npm >/dev/null 2>&1 && npm -v || true
  command -v pnpm >/dev/null 2>&1 && pnpm -v || true
  command -v python3 >/dev/null 2>&1 && python3 --version || true
  command -v pip3 >/dev/null 2>&1 && pip3 --version || true
  command -v latexmk >/dev/null 2>&1 && latexmk -v | head -n 2 || true
  command -v lualatex >/dev/null 2>&1 && lualatex --version | head -n 2 || true
  command -v tlmgr >/dev/null 2>&1 && tlmgr --version | head -n 2 || true
  command -v docker >/dev/null 2>&1 && docker --version || true
  command -v git >/dev/null 2>&1 && git --version || true
  command -v gh >/dev/null 2>&1 && gh --version | head -n 2 || true
  command -v code >/dev/null 2>&1 && code --version | head -n 3 || true
} > "$EXPORT_DIR/system_summary.txt"

apt-mark showmanual | sort > "$EXPORT_DIR/apt-manual.txt"

if [ -f /etc/apt/sources.list ]; then
  cp /etc/apt/sources.list "$EXPORT_DIR/apt-sources/"
fi
cp /etc/apt/sources.list.d/* "$EXPORT_DIR/apt-sources/" 2>/dev/null || true

if command -v code >/dev/null 2>&1; then
  code --list-extensions --show-versions > "$EXPORT_DIR/vscode/extensions.txt"
fi

if [ -f "$HOME/.vscode-server/data/Machine/settings.json" ]; then
  cp "$HOME/.vscode-server/data/Machine/settings.json" "$EXPORT_DIR/vscode/machine-settings.json"
fi

if [ -f "$WORKSPACE_ROOT/.vscode/settings.json" ]; then
  cp "$WORKSPACE_ROOT/.vscode/settings.json" "$EXPORT_DIR/vscode/workspace-settings.json"
fi

if [ -f "/mnt/c/Users/$WIN_USER/AppData/Roaming/Code/User/settings.json" ]; then
  cp "/mnt/c/Users/$WIN_USER/AppData/Roaming/Code/User/settings.json" "$EXPORT_DIR/vscode/windows-settings.json"
fi

echo "Export complete: $EXPORT_DIR"


Export complete: wsl-env-export


## B. Install core packages on target WSL

This uses Ubuntu 24.04 package names for WSL.


In [ ]:
%%bash
set -euo pipefail

sudo apt-get update
sudo apt-get install -y ca-certificates curl gnupg lsb-release

# Docker repo (skip if already configured)
if [ ! -f /etc/apt/keyrings/docker.gpg ]; then
  sudo install -m 0755 -d /etc/apt/keyrings
  curl -fsSL https://download.docker.com/linux/ubuntu/gpg | sudo gpg --dearmor -o /etc/apt/keyrings/docker.gpg
  sudo chmod a+r /etc/apt/keyrings/docker.gpg
fi

if [ ! -f /etc/apt/sources.list.d/docker.list ]; then
  echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] https://download.docker.com/linux/ubuntu $(. /etc/os-release && echo $VERSION_CODENAME) stable" | sudo tee /etc/apt/sources.list.d/docker.list > /dev/null
fi

sudo apt-get update

APT_PACKAGES=(
  build-essential bison flex dwarves libelf-dev libncurses-dev libssl-dev
  curl wget git git-lfs gh openssh-client ca-certificates gnupg
  python3 python3-pip perl libfile-homedir-perl libwww-perl libyaml-tiny-perl
  pandoc ghostscript imagemagick fonts-firacode inotify-tools
)

sudo apt-get install -y "${APT_PACKAGES[@]}"

DOCKER_PACKAGES=(docker-ce docker-ce-cli containerd.io docker-buildx-plugin docker-compose-plugin)
if apt-cache show docker-ce >/dev/null 2>&1; then
  sudo apt-get install -y "${DOCKER_PACKAGES[@]}"
  sudo usermod -aG docker "$USER" || true
fi


## C. TeX Live 2025 (optional, large)

This installs TeX Live from TUG, matching the 2025 tree under /usr/local/texlive/2025. Expect a large download.


In [ ]:
%%bash
set -euo pipefail

: "${EXPORT_DIR:?}"

TL_PROFILE="$EXPORT_DIR/texlive.profile"
cat > "$TL_PROFILE" <<'EOF'
selected_scheme scheme-full
TEXDIR /usr/local/texlive/2025
TEXMFCONFIG ~/.texlive2025/texmf-config
TEXMFVAR ~/.texlive2025/texmf-var
TEXMFHOME ~/texmf
binary_x86_64-linux 1
collection-basic 1
collection-latex 1
collection-latexrecommended 1
collection-luatex 1
collection-fontsrecommended 1
collection-langenglish 1
EOF

mkdir -p /tmp/texlive-install
curl -fsSL https://mirror.ctan.org/systems/texlive/tlnet/install-tl-unx.tar.gz | tar -xz -C /tmp/texlive-install --strip-components=1
sudo /tmp/texlive-install/install-tl -profile "$TL_PROFILE"

if ! grep -q "texlive/2025/bin/x86_64-linux" "$HOME/.profile" 2>/dev/null; then
  echo 'export PATH="/usr/local/texlive/2025/bin/x86_64-linux:$PATH"' >> "$HOME/.profile"
fi

rm -rf /tmp/texlive-install


## D. Node + pnpm via NVM


In [ ]:
%%bash
set -euo pipefail

NVM_VERSION="v0.39.7"
NODE_VERSION="25.2.1"
PNPM_VERSION="10.26.1"

if [ ! -d "$HOME/.nvm" ]; then
  curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/$NVM_VERSION/install.sh | bash
fi

# shellcheck disable=SC1090
. "$HOME/.nvm/nvm.sh"

nvm install "$NODE_VERSION"
nvm alias default "$NODE_VERSION"
corepack enable
npm install -g pnpm@"$PNPM_VERSION"


## E. VS Code extensions + settings (from export)

This restores extensions to the WSL VS Code server and copies settings. Backups are stored under the export folder.


In [ ]:
%%bash
set -euo pipefail

: "${EXPORT_DIR:?}"
: "${WORKSPACE_ROOT:?}"
: "${WIN_USER:?}"

backup_dir="$EXPORT_DIR/vscode/backup-$(date +%Y%m%d-%H%M%S)"
mkdir -p "$backup_dir"

if command -v code >/dev/null 2>&1 && [ -f "$EXPORT_DIR/vscode/extensions.txt" ]; then
  while IFS= read -r ext; do
    [ -z "$ext" ] && continue
    code --install-extension "$ext" --force || true
  done < "$EXPORT_DIR/vscode/extensions.txt"
fi

if [ -f "$EXPORT_DIR/vscode/machine-settings.json" ]; then
  if [ -f "$HOME/.vscode-server/data/Machine/settings.json" ]; then
    cp "$HOME/.vscode-server/data/Machine/settings.json" "$backup_dir/machine-settings.json"
  fi
  mkdir -p "$HOME/.vscode-server/data/Machine"
  cp "$EXPORT_DIR/vscode/machine-settings.json" "$HOME/.vscode-server/data/Machine/settings.json"
fi

if [ -f "$EXPORT_DIR/vscode/workspace-settings.json" ]; then
  if [ -f "$WORKSPACE_ROOT/.vscode/settings.json" ]; then
    cp "$WORKSPACE_ROOT/.vscode/settings.json" "$backup_dir/workspace-settings.json"
  fi
  mkdir -p "$WORKSPACE_ROOT/.vscode"
  cp "$EXPORT_DIR/vscode/workspace-settings.json" "$WORKSPACE_ROOT/.vscode/settings.json"
fi

if [ -f "$EXPORT_DIR/vscode/windows-settings.json" ]; then
  win_settings="/mnt/c/Users/$WIN_USER/AppData/Roaming/Code/User/settings.json"
  if [ -f "$win_settings" ]; then
    cp "$win_settings" "$backup_dir/windows-settings.json"
  fi
  mkdir -p "/mnt/c/Users/$WIN_USER/AppData/Roaming/Code/User"
  cp "$EXPORT_DIR/vscode/windows-settings.json" "$win_settings"
fi

echo "VS Code restore complete. Backups: $backup_dir"


## F. WSL config + cleanup (optional)


In [ ]:
%%bash
set -euo pipefail

# Disable Windows PATH injection (requires WSL restart from Windows)
if [ -w /etc/wsl.conf ]; then
  sudo tee /etc/wsl.conf > /dev/null <<'EOF'
[boot]
systemd=true

[interop]
appendWindowsPath=false
EOF
fi

# Cleanup
sudo apt-get autoremove -y
sudo apt-get clean

# Optional: prune Docker build cache
if command -v docker >/dev/null 2>&1; then
  docker builder prune -f || true
fi


## G. Verify toolchain


In [ ]:
%%bash
set -euo pipefail

command -v node >/dev/null 2>&1 && node -v || true
command -v npm >/dev/null 2>&1 && npm -v || true
command -v pnpm >/dev/null 2>&1 && pnpm -v || true
command -v python3 >/dev/null 2>&1 && python3 --version || true
command -v latexmk >/dev/null 2>&1 && latexmk -v | head -n 2 || true
command -v lualatex >/dev/null 2>&1 && lualatex --version | head -n 2 || true
command -v tlmgr >/dev/null 2>&1 && tlmgr --version | head -n 2 || true
command -v docker >/dev/null 2>&1 && docker --version || true
command -v code >/dev/null 2>&1 && code --version | head -n 3 || true


## H. Dev container (committed)

This repo now includes a .devcontainer directory with a Dockerfile and devcontainer.json. Re-run this cell to regenerate the files from the exported extension list. TeX Live install is optional and heavy; keep it separate if build time is a concern.


In [ ]:
%%bash
set -euo pipefail

: "${WORKSPACE_ROOT:?}"
: "${EXPORT_DIR:?}"

DEVCONTAINER_JSON="$WORKSPACE_ROOT/.devcontainer/devcontainer.json"

if [ ! -f "$DEVCONTAINER_JSON" ]; then
  echo "Missing $DEVCONTAINER_JSON. This repo should include .devcontainer/ files."
  exit 1
fi

if [ ! -f "$EXPORT_DIR/vscode/extensions.txt" ]; then
  echo "Missing $EXPORT_DIR/vscode/extensions.txt. Run Section A first."
  exit 1
fi

python3 - <<'PYCODE'
from pathlib import Path
import re

dev_path = Path("$DEVCONTAINER_JSON")
ext_path = Path("$EXPORT_DIR/vscode/extensions.txt")

exts = [line.strip() for line in ext_path.read_text().splitlines() if line.strip()]
if exts and exts[0].lower().startswith("extensions installed on wsl"):
    exts = exts[1:]

if not exts:
    raise SystemExit("No extensions found in extensions.txt")

text = dev_path.read_text()
ext_lines = [f'        "{ext}",' for ext in exts[:-1]] + [f'        "{exts[-1]}"']
new_block = "\n".join(ext_lines)

pattern = re.compile(r'("extensions"\s*:\s*\[)(.*?)(\n\s*\])', re.DOTALL)
match = pattern.search(text)
if not match:
    raise SystemExit("extensions array not found in devcontainer.json")

prefix, _, suffix = match.group(1), match.group(2), match.group(3)
replacement = prefix + "\n" + new_block + suffix
text = text[:match.start()] + replacement + text[match.end():]

dev_path.write_text(text)
print(f"Updated extensions in {dev_path}")
PYCODE
